# Evaluate Model (gate de calidad)
Evalúa el WER del challenger (versión registrada) y decide si pasa el gate. Reutiliza la lógica del proyecto (`notebooks/utils/00_config`).

**Gate (WER, menor es mejor):** `passed = (wer_challenger <= umbral) OR (wer_challenger <= wer_champion)`
- Si no existe `champion`, solo se evalúa contra el umbral.
- Salida (task values): `passed`, `wer_challenger`, `wer_champion`, `predict_errors`.

In [0]:
%pip install -q jiwer soundfile cactus-compute

In [0]:
%run ../notebooks/utils/00_config

In [0]:
import json, math, mlflow, pandas as pd
from mlflow.tracking import MlflowClient
mlflow.set_registry_uri("databricks-uc")

MODEL_NAME = dbutils.widgets.get("model_name")
MODEL_VERSION = dbutils.widgets.get("model_version")
WER_THRESHOLD = float(dbutils.widgets.get("wer_threshold", "0.30"))
MIN_EVALS = int(dbutils.widgets.get("min_evals", "1"))

client = MlflowClient()

# --- WER del challenger (carga la version registrada y predice sobre audios con referencia) ---
# Si el modelo no puede cargarse/ejecutarse en este compute (p.ej. falta cactus/ffmpeg),
# wer_challenger=NaN => passed=False (graceful, permite que la rama archive se ejecute).
wer_challenger = float("nan")
errors = 0
try:
    model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/{MODEL_VERSION}")
    refs, hyps = [], []
    for p in audio_files():
        ref = read_ref(p)
        if not ref:
            continue
        try:
            out = model.predict(pd.DataFrame({"audio_path": [p]}))[0]
            hyp = json.loads(out)["text"] if isinstance(out, str) else str(out)
        except Exception as e:
            errors += 1
            print("predict error", p, e)
            continue
        refs.append(NORM(ref)); hyps.append(NORM(hyp))
    if len(refs) >= MIN_EVALS:
        wer_challenger = float(jiwer.wer(refs, hyps))
except Exception as e:
    print("model load/eval failed:", e)
    errors = -1

# --- WER del champion (de las metricas del run que lo produjo); None si no hay champion ---
wer_champion = None
try:
    cv = client.get_model_version_by_alias(MODEL_NAME, "champion")
    wer_champion = client.get_run(cv.run_id).data.metrics.get("wer_corpus")
except Exception:
    wer_champion = None  # primer despliegue: no hay champion

# Gate (WER: menor es mejor): pasa si WER <= umbral OR WER <= champion
passed = (not math.isnan(wer_challenger)) and (
    (wer_challenger <= WER_THRESHOLD) or
    (wer_champion is not None and wer_challenger <= wer_champion)
)

print(f"challenger v{MODEL_VERSION}: WER={wer_challenger:.4f} champion_WER={wer_champion} umbral={WER_THRESHOLD} predict_errors={errors} -> passed={passed}")

dbutils.jobs.taskValues.set("passed", bool(passed))
dbutils.jobs.taskValues.set("wer_challenger", float(wer_challenger))
dbutils.jobs.taskValues.set("wer_champion", float(wer_champion) if wer_champion is not None else -1.0)
dbutils.jobs.taskValues.set("predict_errors", int(errors))